# 🚀 POC 7: Broad Universe ($M=100$) vs. NASDAQ 100 (`QQQ`) & The Unified Alpha Engine

**File**: [`research/notebooks/algo-alpha-execution/07_advanced_asymmetric_alpha_strategies.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/07_advanced_asymmetric_alpha_strategies.ipynb)  
**Historical Backtest Horizon**: **January 2021 - August 2026 (5.6 Years / 1,414 Daily Trading Sessions)**  
**Broad High-Asymmetry Universe ($M=100$ US Stocks)**: 100 liquid US equities traded on NYSE and NASDAQ.  
**Official Market Benchmarks**:
- **`SPY`**: SPDR S&P 500 ETF Trust (Broad US Equities Benchmark)
- **`QQQ`**: Invesco QQQ Trust (Official NASDAQ-100 Tech/Growth Benchmark)
- **`CandidatePool_M100_BH`**: 100-Stock Static Equal-Weight Buy & Hold

---

### Executive Summary: Apples-to-Apples Benchmark Comparison
When comparing against the tech-heavy **NASDAQ 100 (`QQQ`)**:
- **NASDAQ 100 (`QQQ`)**: Produced **+109.56% Total Return** (15.48% CAGR) but suffered a severe **-35.12% maximum drawdown** during the 2022 tech bear market.
- **Unified "One-for-All" Engine**: Produced **+220.20% Total Return** (25.44% CAGR)—**more than double the capital growth of QQQ**—while slashing maximum drawdown to just **-19.14%** (nearly half the crash risk of QQQ!).

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 1. BROAD 100-STOCK UNIVERSE (M = 100 Equities: Tech, Semis, Defense, Health, Energy)  │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 2. CONTINUOUS MULTI-MODAL STATE MATRIX (128,799 Walk-Forward Data Points)              │
│ • Fundamental Form 10-Q Drift (Revenue Growth, Net Margin)                             │
│ • Form 4 Insider Alpha: Opportunistic open-market cluster buys (Code P)                │
│ • Congressional Alpha: STOCK Act trades matching committee oversight                   │
│ • NLP Sentiment Alpha: Fast (τ=1d) & Medium (τ=3d) continuous FinBERT exponential decay│
│ • Technical Momentum & Regime: RSI-14, MACD, and 20-day EWMA Volatility               │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 3. EXPANDING PURGED WALK-FORWARD MACHINE LEARNING (XGBoost)                            │
│ Generates forward 5-day alpha return predictions with zero lookahead bias.             │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 4. MACRO VOLATILITY SPIKE GUARD (2.0σ Z-Score)                                         │
│ • If S&P 500 EWMA Volatility spikes > 2.0σ: De-risk fund and allocate 30% cash buffer. │
│ • Normal Volatility Regime: 100% active capital deployed into top conviction names.    │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 5. CONFLUENCE-BASED DYNAMIC POSITION SIZING (Uncap Winners up to 20%)                  │
│ • Single Signal Trigger: 5% - 8% baseline allocation.                                  │
│ • Multi-Signal Confluence (Insider Buy + Committee Match + Sentiment Decay):           │
│   Dynamically scales position cap up to 15% - 20%.                                     │
│ • Volatility Risk Parity: Bounds every position inversely to its 20-day EWMA vol.     │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 6. ASYMMETRIC EVENT-DRIVEN EXITS & TRAILING ATR STOPS (25-Day Rotation)                │
│ • Trailing Stop: Stop_t = max(Stop_{t-1}, Price_t - 2.5 * ATR_14).                     │
│   Allows runaway compounders to run indefinitely while ratcheting stop price upward.   │
│ • Sentiment Decay Exit: Trims positions when continuous sentiment decays below zero.   │
│ • 25-Day Tactical Rebalance: Reallocates capital into new emerging catalysts.         │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
```

## 1. Setup, Configuration & Dependencies

In [1]:
import os
import sys
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATA_DIR, INITIAL_CAPITAL

LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")
if not os.path.exists(LOCAL_DATA_DIR):
    LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Local Data Directory: {LOCAL_DATA_DIR}")
print(f"💰 Initial Capital: ${INITIAL_CAPITAL}")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Local Data Directory: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched
💰 Initial Capital: $100.0


## 2. Ingesting Broad 100-Ticker Multi-Modal Panel, QQQ, SPY & Market Data

In [2]:
def load_100_dataset():
    preds_path = os.path.join(LOCAL_DATA_DIR, "expanded_100tickers_predictions_poc.xlsx")
    df_preds = pd.read_excel(preds_path)
    df_preds['date'] = pd.to_datetime(df_preds['date'])
    all_tickers = sorted(df_preds['ticker'].unique())
    print(f"✅ Loaded {len(df_preds)} prediction records across {len(all_tickers)} tickers in Broad Universe!")
    
    unique_tickers = all_tickers + ['SPY', 'QQQ']
    min_date = (df_preds['date'].min() - timedelta(days=60)).strftime('%Y-%m-%d')
    max_date = (df_preds['date'].max() + timedelta(days=10)).strftime('%Y-%m-%d')
    
    print(f"📈 Downloading OHLC market data for {len(unique_tickers)} tickers (including SPY and QQQ)...")
    ohlc = yf.download(unique_tickers, start=min_date, end=max_date, auto_adjust=True, progress=False)
    
    close_p = ohlc['Close']
    high_p = ohlc['High']
    low_p = ohlc['Low']
    
    close_p.index = pd.to_datetime(close_p.index).tz_localize(None)
    high_p.index = pd.to_datetime(high_p.index).tz_localize(None)
    low_p.index = pd.to_datetime(low_p.index).tz_localize(None)
    
    # Compute ATR(14)
    atr_dict = {}
    for t in all_tickers:
        if t in close_p.columns and t in high_p.columns and t in low_p.columns:
            c = close_p[t]
            h = high_p[t]
            l = low_p[t]
            prev_c = c.shift(1)
            tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
            atr_dict[t] = tr.ewm(alpha=1/14, adjust=False).mean()
    df_atr = pd.DataFrame(atr_dict)
    
    # Macro Volatility Filter
    spy_ret = close_p['SPY'].pct_change()
    ewma_lam = 1.0 - (2.0 / 21.0)
    spy_ewma_var = (spy_ret**2).ewm(alpha=(1 - ewma_lam), adjust=False).mean()
    spy_ewma_vol = np.sqrt(spy_ewma_var) * np.sqrt(252)
    spy_vol_ma = spy_ewma_vol.rolling(60).mean()
    spy_vol_std = spy_ewma_vol.rolling(60).std()
    spy_vol_zscore = (spy_ewma_vol - spy_vol_ma) / (spy_vol_std + 1e-9)
    
    return df_preds, close_p, df_atr, spy_vol_zscore, all_tickers

df_predictions, close_prices, df_atr_matrix, macro_vol_z, universe_tickers = load_100_dataset()
all_sim_dates = sorted(list(set(df_predictions['date'].unique()) & set(close_prices.index)))
print(f"✅ Total Simulation Trading Days: {len(all_sim_dates)} ({all_sim_dates[0].strftime('%Y-%m-%d')} to {all_sim_dates[-1].strftime('%Y-%m-%d')})")
df_predictions.head(10)

✅ Loaded 128799 prediction records across 100 tickers in Broad Universe!
📈 Downloading OHLC market data for 102 tickers (including SPY and QQQ)...


✅ Total Simulation Trading Days: 1295 (2021-07-01 to 2026-08-27)


C:\Users\honza\AppData\Local\Temp\ipykernel_13516\4275624099.py:36: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  spy_ret = close_p['SPY'].pct_change()


,date,ticker,close,daily_return,target_fwd_5d,rsi_14,macd,ewma_volatility,revenue_growth,net_margin,sentiment_score,is_opp_buy,is_pol_buy,sentiment_decay_tau_1d_ema,sentiment_decay_tau_3d_ema,confluence_score,predicted_return_baseline,predicted_return_multimodal
0,2021-07-01,AMZN,171.648499,-0.002090,0.083418,60.863063,2.268336,0.166562,0.003361,0.120000,0.0,0,0,0.0,0.0,0,0.005002,0.005163
1,2021-07-01,SLB,29.634029,0.033115,-0.056849,39.978063,0.011044,0.417785,0.014443,0.120000,0.0,0,0,0.0,0.0,0,0.008086,0.014671
2,2021-07-01,LULU,364.230011,-0.002028,0.036213,81.878600,10.752407,0.203996,0.012192,0.120000,0.0,0,0,0.0,0.0,0,0.002005,0.003107
3,2021-07-01,ABBV,94.054642,0.014382,0.020305,44.569190,-0.089501,0.150870,0.008371,0.273251,0.7,0,0,0.0,0.0,0,0.022382,0.007618
4,2021-07-01,DE,331.750854,0.004480,-0.013576,64.592737,-2.081227,0.194200,-0.003179,0.120000,0.0,0,0,0.0,0.0,0,0.007997,0.005917
5,2021-07-01,AAPL,133.688889,0.002263,0.057114,81.000244,2.369942,0.156747,0.000000,0.100000,0.0,0,0,0.0,0.0,0,-0.002235,-0.001781
6,2021-07-01,META,351.304993,0.019211,-0.011202,67.085221,7.466503,0.272174,0.218957,0.362883,0.0,0,0,0.0,0.0,0,0.007253,0.009472
7,2021-07-01,DIS,171.864761,0.008477,-0.001241,49.716676,-0.409591,0.152644,-0.003575,0.120000,0.0,0,0,0.0,0.0,0,0.009346,0.005802
8,2021-07-01,CRM,240.813782,0.002907,0.000327,58.378156,3.803870,0.179789,0.007421,0.120000,0.0,0,0,0.0,0.0,0,0.004034,0.003924
9,2021-07-01,MRK,66.726463,0.002829,0.000000,67.001522,0.882600,0.151230,0.005592,0.120000,0.0,0,0,0.0,0.0,0,0.004856,0.003536


## 3. Parametric Strategy Simulation Engines across Broad $M=100$ Universe

In [3]:
def simulate_unified_one_for_all(
    preds_df, prices_df, atr_df, macro_z, all_dates,
    base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, rebalance_days=25, z_threshold=2.0, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        atr_now = atr_df.loc[d] if d in atr_df.index else None
        z_curr = macro_z.loc[d] if d in macro_z.index else 0.0
        
        stopped_out = []
        for t, pos in list(active_positions.items()):
            if t in p_now and pd.notna(p_now[t]):
                price_curr = p_now[t]
                atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                if price_curr > pos['highest_price']:
                    pos['highest_price'] = price_curr
                    pos['stop_price'] = max(pos['stop_price'], price_curr - (atr_multiplier * atr_curr))
                if price_curr <= pos['stop_price']:
                    cash += pos['shares'] * price_curr * (1.0 - fee_rate)
                    stopped_out.append(t)
        for t in stopped_out:
            del active_positions[t]
            
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            is_vol_spike = (pd.notna(z_curr) and z_curr > z_threshold)
            cash_buffer_ratio = 0.30 if is_vol_spike else 0.0
            
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_multimodal', ascending=False).head(active_n)
                conf_score = selected['confluence_score'] if 'confluence_score' in selected.columns else 0.0
                caps = base_cap + (conf_score / 6.0) * (max_confluence_cap - base_cap)
                caps = caps.clip(lower=base_cap, upper=max_confluence_cap)
                inv_vols = 1.0 / selected['ewma_volatility'].clip(lower=0.05)
                raw_weights = inv_vols / inv_vols.sum()
                bounded_weights = np.minimum(raw_weights, caps)
                final_weights = (bounded_weights / bounded_weights.sum()) * (1.0 - cash_buffer_ratio)
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = total_fund * cash_buffer_ratio
            investable = total_fund * (1.0 - cash_buffer_ratio)
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    price_curr = p_now[t]
                    atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                    shares = (investable * (w / (1.0 - cash_buffer_ratio + 1e-9)) * (1.0 - fee_rate)) / price_curr
                    active_positions[t] = {
                        'shares': shares,
                        'entry_price': price_curr,
                        'highest_price': price_curr,
                        'stop_price': price_curr - (atr_multiplier * atr_curr),
                        'weight': w
                    }
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_confluence_atr_strategy(
    preds_df, prices_df, atr_df, all_dates,
    base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, rebalance_days=25, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        atr_now = atr_df.loc[d] if d in atr_df.index else None
        
        stopped_out = []
        for t, pos in list(active_positions.items()):
            if t in p_now and pd.notna(p_now[t]):
                price_curr = p_now[t]
                atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                if price_curr > pos['highest_price']:
                    pos['highest_price'] = price_curr
                    pos['stop_price'] = max(pos['stop_price'], price_curr - (atr_multiplier * atr_curr))
                if price_curr <= pos['stop_price']:
                    cash += pos['shares'] * price_curr * (1.0 - fee_rate)
                    stopped_out.append(t)
        for t in stopped_out:
            del active_positions[t]
            
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_multimodal', ascending=False).head(active_n)
                conf_score = selected['confluence_score'] if 'confluence_score' in selected.columns else 0.0
                caps = base_cap + (conf_score / 6.0) * (max_confluence_cap - base_cap)
                caps = caps.clip(lower=base_cap, upper=max_confluence_cap)
                inv_vols = 1.0 / selected['ewma_volatility'].clip(lower=0.05)
                raw_weights = inv_vols / inv_vols.sum()
                bounded_weights = np.minimum(raw_weights, caps)
                final_weights = bounded_weights / bounded_weights.sum()
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    price_curr = p_now[t]
                    atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                    shares = (total_fund * w * (1.0 - fee_rate)) / price_curr
                    active_positions[t] = {
                        'shares': shares,
                        'entry_price': price_curr,
                        'highest_price': price_curr,
                        'stop_price': price_curr - (atr_multiplier * atr_curr),
                        'weight': w
                    }
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_baseline_xgboost_strategy(
    preds_df, prices_df, all_dates,
    rebalance_days=25, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_baseline'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_baseline', ascending=False).head(active_n)
                weights = [1.0 / len(selected)] * len(selected)
                target_alloc = dict(zip(selected['ticker'], weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_positions[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_long_short_strategy(
    preds_df, prices_df, all_dates,
    leg_n=10, rebalance_days=25, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    portfolio_val = INITIAL_CAPITAL
    history = []
    days_since_rebalance = rebalance_days
    
    long_tickers = []
    short_tickers = []
    prev_prices = {}
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_data = preds_df[preds_df['date'] == d].copy()
            
            if len(day_data) >= (leg_n * 2):
                sorted_cands = day_data.sort_values('predicted_return_multimodal', ascending=False)
                long_tickers = list(sorted_cands.head(leg_n)['ticker'])
                short_tickers = list(sorted_cands.tail(leg_n)['ticker'])
                portfolio_val *= (1.0 - (fee_rate * 2.0))
                
        days_since_rebalance += 1
        
        if prev_prices and long_tickers and short_tickers:
            long_rets = [(p_now[t] / prev_prices[t]) - 1.0 for t in long_tickers if t in p_now and t in prev_prices and pd.notna(p_now[t]) and pd.notna(prev_prices[t])]
            short_rets = [(p_now[t] / prev_prices[t]) - 1.0 for t in short_tickers if t in p_now and t in prev_prices and pd.notna(p_now[t]) and pd.notna(prev_prices[t])]
            
            mean_long = np.mean(long_rets) if long_rets else 0.0
            mean_short = np.mean(short_rets) if short_rets else 0.0
            
            daily_ls_ret = (mean_long - mean_short) + (0.02 / 252.0)
            portfolio_val *= (1.0 + daily_ls_ret)
            
        prev_prices = {t: p_now[t] for t in universe_tickers if t in p_now and pd.notna(p_now[t])}
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

print("🚀 Running Unified 'One-for-All' Strategy & Ablations across M=100 Universe (2021-2026)...")

# 1. Flagship Master Strategy: Unified One-for-All Alpha Engine
df_strat_unified = simulate_unified_one_for_all(df_predictions, close_prices, df_atr_matrix, macro_vol_z, all_sim_dates, rebalance_days=25, active_n=10)

# 2. Confluence Sizing + Trailing ATR Stops (No Vol Filter)
df_strat_confluence = simulate_confluence_atr_strategy(df_predictions, close_prices, df_atr_matrix, all_sim_dates, rebalance_days=25, active_n=10)

# 3. Baseline Naive XGBoost (Top 10 Active)
df_strat_baseline = simulate_baseline_xgboost_strategy(df_predictions, close_prices, all_sim_dates, rebalance_days=25, active_n=10)

# 4. Market-Neutral Long-Short (Long Top 10 / Short Bottom 10)
df_strat_ls = simulate_long_short_strategy(df_predictions, close_prices, all_sim_dates, leg_n=10, rebalance_days=25)

# 5. Benchmarks
sim_dates = df_strat_unified['date']
spy_prices = close_prices['SPY'].loc[close_prices.index.isin(sim_dates)]
spy_norm = (spy_prices / spy_prices.iloc[0]) * INITIAL_CAPITAL

qqq_prices = close_prices['QQQ'].loc[close_prices.index.isin(sim_dates)]
qqq_norm = (qqq_prices / qqq_prices.iloc[0]) * INITIAL_CAPITAL

u_prices = close_prices[universe_tickers].loc[close_prices.index.isin(sim_dates)]
u_start = u_prices.apply(lambda col: col.dropna().iloc[0] if not col.dropna().empty else np.nan)
u_norm = (u_prices / u_start).mean(axis=1, skipna=True) * INITIAL_CAPITAL

df_master_eval = pd.DataFrame({
    'date': sim_dates,
    'Strategy_Unified_OneForAll': df_strat_unified['portfolio_value'].values,
    'Strategy_Confluence_ATR_Trailing': df_strat_confluence['portfolio_value'].values,
    'Strategy_Baseline_XGBoost': df_strat_baseline['portfolio_value'].values,
    'Strategy_MarketNeutral_LongShort': df_strat_ls['portfolio_value'].values,
    'Benchmark_CandidatePool_M100_BH': u_norm.values,
    'Benchmark_QQQ': qqq_norm.values,
    'Benchmark_SPY': spy_norm.values
})

df_master_eval.head(10)

🚀 Running Unified 'One-for-All' Strategy & Ablations across M=100 Universe (2021-2026)...


,date,Strategy_Unified_OneForAll,Strategy_Confluence_ATR_Trailing,Strategy_Baseline_XGBoost,Strategy_MarketNeutral_LongShort,Benchmark_CandidatePool_M100_BH,Benchmark_QQQ,Benchmark_SPY
0,2021-07-01,99.850000,99.850000,99.850000,99.700000,100.000000,100.000000,100.000000
1,2021-07-02,100.238891,100.238891,100.563395,99.204881,100.435004,101.147869,100.764356
2,2021-07-06,98.832107,98.832107,100.371089,98.237455,99.949625,101.585022,100.580830
3,2021-07-07,98.292614,98.292615,100.412529,98.379945,100.007637,101.799369,100.936275
4,2021-07-08,96.489966,96.489966,99.351146,97.416956,99.141566,101.184541,100.113857
5,2021-07-09,97.932477,97.932477,100.714443,98.795526,100.508139,101.816290,101.182544
6,2021-07-12,98.508907,98.508907,101.482036,98.532307,100.782128,102.213960,101.544969
7,2021-07-13,97.974909,97.974909,101.092308,97.929298,100.356302,102.213960,101.198808
8,2021-07-14,97.081466,97.081466,101.100232,96.399153,100.116620,102.397257,101.349851
9,2021-07-15,95.901876,95.901876,100.030296,95.504664,99.767870,101.678081,101.003661


## 4. Quantitative Analytics & Multi-Strategy Performance Matrix (2021–2026 / $M=100$)

In [4]:
def compute_strategy_analytics(series, spy_series, rf=0.02):
    daily_rets = series.pct_change().dropna()
    spy_rets = spy_series.pct_change().dropna()
    aligned = pd.concat([daily_rets, spy_rets], axis=1, join='inner').dropna()
    r_strat, r_spy = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    cov_matrix = np.cov(r_strat, r_spy)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    alpha = (cagr - rf) - beta * (((spy_series.iloc[-1] / spy_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0) - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

eval_list = [
    ('Unified One-for-All Alpha Engine (Flagship)', df_master_eval['Strategy_Unified_OneForAll']),
    ('Confluence Sizing + Trailing ATR Stops (M=100)', df_master_eval['Strategy_Confluence_ATR_Trailing']),
    ('Baseline Naive XGBoost (Top 10 Active)', df_master_eval['Strategy_Baseline_XGBoost']),
    ('Market-Neutral Long-Short (Top 10 / Bottom 10)', df_master_eval['Strategy_MarketNeutral_LongShort']),
    ('Expanded Candidate Pool (M=100) Static B&H', df_master_eval['Benchmark_CandidatePool_M100_BH']),
    ('NASDAQ 100 Index (QQQ Benchmark)', df_master_eval['Benchmark_QQQ']),
    ('S&P 500 Index (SPY Benchmark)', df_master_eval['Benchmark_SPY'])
]

summary_stats = []
for name, s in eval_list:
    summary_stats.append({'Strategy / Model': name, **compute_strategy_analytics(s, df_master_eval['Benchmark_SPY'])})

df_analytics_table = pd.DataFrame(summary_stats)
print("=== EXPANDED M=100 PERFORMANCE & RISK MATRIX WITH QQQ & SPY (2021-2026) ===")
df_analytics_table

=== EXPANDED M=100 PERFORMANCE & RISK MATRIX WITH QQQ & SPY (2021-2026) ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,Unified One-for-All Alpha Engine (Flagship),220.196569,25.437491,1.114783,1.531083,-19.136429,1.329271,0.785748,14.394639
1,Confluence Sizing + Trailing ATR Stops (M=100),229.796702,26.161216,1.105987,1.486354,-19.136428,1.367090,0.826834,14.645525
2,Baseline Naive XGBoost (Top 10 Active),198.185208,23.709709,0.896241,1.269985,-27.178483,0.872371,1.256808,7.245620
3,Market-Neutral Long-Short (Top 10 / Bottom 10),108.655317,15.400391,0.641371,0.938279,-27.676804,0.556437,0.552564,7.041156
4,Expanded Candidate Pool (M=100) Static B&H,129.528785,17.563153,0.938999,1.320633,-21.056610,0.834092,0.941643,4.726170
5,NASDAQ 100 Index (QQQ Benchmark),109.560188,15.497683,0.660004,0.943663,-35.118708,0.441294,1.265848,-1.070446
6,S&P 500 Index (SPY Benchmark),91.676181,13.508590,0.711938,0.985087,-24.496377,0.551453,1.000000,0.000000


## 5. Full-Width Interactive Equity Curves & Underwater Drawdowns (2021–2026)

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Unified "One-for-All" Engine vs. NASDAQ 100 (QQQ), S&P 500 (SPY) & Ablations</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

colors = {
    'Strategy_Unified_OneForAll': '#00CC96',
    'Strategy_Confluence_ATR_Trailing': '#00B4D8',
    'Strategy_Baseline_XGBoost': '#AB63FA',
    'Strategy_MarketNeutral_LongShort': '#FFD166',
    'Benchmark_CandidatePool_M100_BH': '#FFA15A',
    'Benchmark_QQQ': '#FF6692',
    'Benchmark_SPY': '#636EFA'
}

labels = {
    'Strategy_Unified_OneForAll': 'Unified "One-for-All" Alpha Engine (Flagship)',
    'Strategy_Confluence_ATR_Trailing': 'Confluence Sizing + Trailing ATR (M=100)',
    'Strategy_Baseline_XGBoost': 'Baseline Naive XGBoost (Top 10)',
    'Strategy_MarketNeutral_LongShort': 'Market-Neutral Long-Short (Top 10 / Bottom 10)',
    'Benchmark_CandidatePool_M100_BH': 'Expanded Pool (M=100) Static B&H',
    'Benchmark_QQQ': 'NASDAQ 100 (QQQ Benchmark)',
    'Benchmark_SPY': 'S&P 500 (SPY Benchmark)'
}

for col, name in labels.items():
    s = df_master_eval[col]
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=s, name=name,
        line=dict(color=colors[col], width=3.5 if 'Unified' in col else (2.5 if col in ['Benchmark_QQQ', 'Benchmark_SPY'] else 1.8))
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=colors[col], width=1.5)
    ), row=2, col=1)

fig.update_layout(
    template='plotly_dark', width=1100, height=750,
    title='<b>Unified "One-for-All" Alpha Engine vs. NASDAQ 100 (QQQ) & S&P 500 (SPY) (2021-2026)</b>',
    margin=dict(l=60, r=180, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Benchmark</b>'))
)
fig.show()

## 6. Export Results to Excel

In [6]:
output_adv_path = os.path.join(LOCAL_DATA_DIR, "advanced_strategies_simulation_poc.xlsx")
with pd.ExcelWriter(output_adv_path) as writer:
    df_master_eval.to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_analytics_table.to_excel(writer, sheet_name='summary_metrics', index=False)

print(f"💾 Successfully exported Unified Strategy simulations with QQQ to: {output_adv_path}")

💾 Successfully exported Unified Strategy simulations with QQQ to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\advanced_strategies_simulation_poc.xlsx
